# 03 - Reason-for-recall text EDA (T3.1.1)
Logic lives in `fda_recall.analysis.text_eda`. Purpose: understand phrasing so the taxonomy and keyword baseline are written from data **without** looking at gold labels.

In [1]:
import pandas as pd

from fda_recall.analysis import text_eda as te

ev = pd.read_parquet(te.ROOT / 'data/processed/recalls_event.parquet')
ev.groupby('product_type').size()

product_type
Drugs    4740
Food     7947
dtype: int64

## Distinct texts and near-duplicates
`near_dup_rate` = 1 - (distinct texts after stripping digits/punctuation) / events.

In [ ]:
s = te.summary(ev)
s

## Length distribution

In [3]:
te.plot_lengths(ev)

WindowsPath('C:/Users/Gideon/resume_projects/fda-recall-quality-risk-dashboard/reports/figures/text_eda_length.png')

## Most common 20-character prefixes

In [4]:
te.top_prefixes(ev)

,prefix,events
0,lack of assurance of,440
1,failed impurities/de,379
2,failed dissolution s,300
3,presence of particul,245
4,marketed without an,243
5,presence of foreign,236
6,product contains und,233
7,product may contain,147
8,microbial contaminat,146
9,the firm was notifie,135


In [5]:
for pt in ['Drugs','Food']:
    display(pt, te.top_prefixes(ev[ev.product_type==pt], n=20))

'Drugs'

,prefix,events
0,lack of assurance of,440
1,failed impurities/de,379
2,failed dissolution s,300
3,presence of particul,245
4,marketed without an,243
5,presence of foreign,233
6,microbial contaminat,143
7,labeling: incorrect,126
8,cgmp deviations,116
9,defective container:,95


'Food'

,prefix,events
0,product contains und,233
1,product may contain,147
2,the firm was notifie,135
3,potential contaminat,132
4,product may be conta,123
5,undeclared allergens,98
6,undeclared allergen,91
7,possible contaminati,60
8,potential salmonella,58
9,contains undeclared,55


## Top unigrams and bigrams

In [6]:
paths = [te.plot_ngrams(ev, pt, ng, lab) for pt in ['Drugs','Food'] for ng, lab in [((1,1),'unigrams'),((2,2),'bigrams')]]
paths

[WindowsPath('C:/Users/Gideon/resume_projects/fda-recall-quality-risk-dashboard/reports/figures/text_eda_unigrams_drugs.png'),
 WindowsPath('C:/Users/Gideon/resume_projects/fda-recall-quality-risk-dashboard/reports/figures/text_eda_bigrams_drugs.png'),
 WindowsPath('C:/Users/Gideon/resume_projects/fda-recall-quality-risk-dashboard/reports/figures/text_eda_unigrams_food.png'),
 WindowsPath('C:/Users/Gideon/resume_projects/fda-recall-quality-risk-dashboard/reports/figures/text_eda_bigrams_food.png')]

In [7]:
for pt in ['Drugs','Food']:
    print(pt, te.top_ngrams(ev.loc[ev.product_type==pt,'reason_norm'],(2,2),15).to_dict())

Drugs {'cgmp deviations': 654, 'assurance sterility': 453, 'lack assurance': 453, 'impurities degradation': 411, 'failed impurities': 405, 'degradation specifications': 380, 'particulate matter': 316, 'failed dissolution': 306, 'dissolution specifications': 271, 'presence particulate': 261, 'presence foreign': 260, 'approved nda': 257, 'nda anda': 257, 'marketed approved': 256, 'specifications specification': 232}
Food {'listeria monocytogenes': 898, 'product contains': 517, 'undeclared milk': 491, 'contains undeclared': 475, 'contain undeclared': 388, 'contaminated salmonella': 376, 'contaminated listeria': 373, 'potential contaminated': 362, 'undeclared allergen': 287, 'potential contamination': 259, 'product contain': 247, 'firm notified': 235, 'product contaminated': 200, 'undeclared wheat': 195, 'undeclared soy': 195}


## Coverage check against taxonomy example phrases
Share of events matching at least one taxonomy example phrase (rough signal of taxonomy coverage, not a classifier).

In [8]:
import re

import yaml

tax = yaml.safe_load(open(te.ROOT/'data/reference/taxonomy.yaml', encoding='utf-8'))
pat = '|'.join(re.escape(e) for c in tax['categories'] if c['id']!='OTHER' for e in c['examples'])
ev.assign(hit=ev.reason_norm.str.contains(pat)).groupby('product_type').hit.mean()

product_type
Drugs    0.725316
Food     0.125204
Name: hit, dtype: float64